## 3D (multi-view) Hummingbird evaluation on MVImgNet

Builds the patch memory from selected viewpoint (angle) bins of MVImgNet and runs the dense nearest-neighbour retrieval evaluation separately on each validation viewpoint bin, giving per-class IoU as a function of viewpoint change.

<a href="https://githubtocolab.com/ToyeshC/open-hummingbird-3d-eval/blob/add-3d-evaluation/examples/hbird_3d_eval_example_new.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

### 1. Install required Libraries

In [ ]:
# ---- PyTorch (CUDA 12.1) ----
!pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu121
!pip install lightning==2.4.0
!pip install torchmetrics==1.7.0
!pip install tqdm==4.67.1  # for progress bars
!pip install scipy==1.15.2
!pip install joblib==1.4.2
!pip install numpy==1.26.4  # torch bundles the correct triton
# !pip install faiss-gpu-cu12  # CUDA-specific wheel may mismatch the Colab runtime
# !pip install faiss-gpu  # not available for Python 3.12 in Colab
!pip install faiss-cpu  # CPU FAISS since Colab runs Python 3.12; original experiments used Python 3.11 with faiss-gpu-cu12
!pip install -q gdown  # for the MVImgNet subset
!pip install pyyaml  # required for ModelCheckpoint in pytorch_lightning
!pip install transformers==4.51.3  # needed for HF models; 5.x requires torch>=2.4 and matches the version used for the paper results

**Important**: After the installation step please restart the Runtime/Kernel before continuing with the step 2.

### 2. Access MVImgNet subset and clone our repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



In [ ]:
# Path to the MVImgNet subset split into angle bins, with structure:
#   <class_id>/<angle_bin>/img/*.jpg and <class_id>/<angle_bin>/mask/*.png
# as produced by examples/mvimgnet_create_bins.ipynb
# TODO before the PR: replace the Drive mount with a public download (gdown) link
data_dir = '/content/drive/MyDrive/datasets/mvimgnet'
!ls "{data_dir}"

In [ ]:
!git clone https://github.com/ToyeshC/open-hummingbird-3d-eval.git

In [ ]:
# Move to the repository folder
%cd open-hummingbird-3d-eval

# Checkout to branch with multiview evaluation
!git checkout add-3d-evaluation

### 3. Unzip Contents of zip Dataset

In [ ]:
# Optional: if the subset is stored as a zip in Drive, copying it to the local
# disk and unzipping there is much faster than reading through the Drive mount
# !cp "/content/drive/MyDrive/datasets/mvimgnet.zip" /content/
# !unzip -q /content/mvimgnet.zip -d /content/mvimgnet
# data_dir = '/content/mvimgnet'

### 4. Install repo

In [ ]:
!pip install .

### 5. Evaluate a preferred model on the downloaded dataset

In [ ]:
import torch
from hbird.hbird_eval import hbird_evaluation

In [ ]:
# Parameters for the model dino
device = 'cuda'
input_size = 512
batch_size = 4
patch_size = 16
embed_dim = 384  # dino_vits16 has 384-dim embeddings
model = torch.hub.load('facebookresearch/dino:main', 'dino_vits16')
num_workers = 8
n_neighbours = 30
nn_method = 'faiss'
memory_size = 1024000
augmentation_epoch = 1

In [ ]:
# Dataset configuration (data_dir is set in step 2)
dataset_name = 'mvimgnet'

# The memory is built from train_bins; each bin in val_bins is evaluated separately
train_bins = ['0']
val_bins = ['0', '45', '90']

In [ ]:
def extract_dino_features(model, imgs):
    return model.get_intermediate_layers(imgs)[0][:, 1:], None

In [ ]:
import numpy as np

miou_per_bin = hbird_evaluation(model.to(device),
        d_model=embed_dim,        # size of the embedding feature vectors of patches
        patch_size=patch_size,
        batch_size=batch_size,
        input_size=input_size,
        augmentation_epoch=augmentation_epoch,  # augmentation iterations over the training set when building the memory
        device=device,
        return_knn_details=False,
        nn_method=nn_method,
        n_neighbours=n_neighbours,  # neighbours fetched per image patch
        nn_params=None,
        ftr_extr_fn=extract_dino_features,  # extracts patch features from the vision encoder
        dataset_name=dataset_name,
        data_dir=data_dir,
        memory_size=memory_size,
        num_workers=num_workers,
        train_bins=train_bins,
        val_bins=val_bins)

for val_bin, per_class_iou in zip(val_bins, miou_per_bin):
    print(f"val bin {val_bin:>3}: mean mIoU = {np.mean(per_class_iou):.4f}")

### 6. Optional: full experiment sweep (Experiments A and B)

Reproduces the paper's viewpoint-generalization sweep: for every list of training bins the memory is rebuilt and all validation bins are evaluated. Results are appended to a CSV compatible with `hbird_eval_multiview_analysis_memory.ipynb`. This runs the full evaluation once per training-bin list, so expect it to take a while.

In [ ]:
import csv
import os
from tqdm import tqdm

RESULTS_PATH = 'results/results_exp_a_b.csv'
VAL_BINS = ['0', '15', '30', '45', '60', '75', '90']
TRAIN_BIN_LISTS = [
    ['0', '30', '60', '90'],
    ['0', '45', '90'],
    ['0', '90'],
    ['0'],
]

os.makedirs('results', exist_ok=True)
if not os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, 'w') as f:
        f.write('job_id,model,train_bins,val_bin,jac_mean,jac_std,'
                + ','.join(f'jac{i}' for i in range(16))
                + ',d_model,batch_size,input_size,patch_size\n')

for sweep_train_bins in tqdm(TRAIN_BIN_LISTS):
    miou_per_bin = hbird_evaluation(model.to(device),
            d_model=embed_dim,
            patch_size=patch_size,
            batch_size=batch_size,
            input_size=input_size,
            augmentation_epoch=augmentation_epoch,
            device=device,
            nn_method=nn_method,
            n_neighbours=n_neighbours,
            ftr_extr_fn=extract_dino_features,
            dataset_name=dataset_name,
            data_dir=data_dir,
            memory_size=memory_size,
            num_workers=num_workers,
            train_bins=sweep_train_bins,
            val_bins=VAL_BINS)

    train_str = '_'.join(sweep_train_bins)
    with open(RESULTS_PATH, 'a', newline='') as f:
        writer = csv.writer(f)
        for val_bin, per_class_iou in zip(VAL_BINS, miou_per_bin):
            writer.writerow(['colab', 'dino_vits16', train_str, val_bin,
                             round(float(np.mean(per_class_iou)), 3),
                             round(float(np.std(per_class_iou)), 3),
                             *[round(float(x), 3) for x in per_class_iou],
                             embed_dim, batch_size, input_size, patch_size])
    print(f'Saved results for train_bins={train_str}')